# SSN optimizer

This notebook shows the optimizer surface used by PDAP.

The important pieces are:

- the outer-weight parameter vector,
- the data Hessian provided by the model closure,
- the masked proximal structure (`penalized_mask` and `nonneg_mask`), and
- the globalization method (`levenberg_marquardt` or `steihaug_cg`).


In [1]:
import torch

from src.SSN.optimizer import SSN

w = torch.nn.Parameter(torch.tensor([0.0], dtype=torch.float64))
optimizer = SSN([w], alpha=0.0, gamma=0.0, power=1.0)
optimizer.data_hessian = torch.eye(1, dtype=torch.float64)

def closure():
    optimizer.zero_grad()
    return 0.5 * (w - 1.0).pow(2).sum()

before = float(w.item())
loss = optimizer.step(closure)
after = float(w.item())

print({"before": before, "after": after, "loss": float(loss.detach())})


{'before': 0.0, 'after': 1.0, 'loss': 0.0}


The real PDAP training loop uses the same optimizer class through the model
closure: the model constructs the data Hessian, builds the closure, then
calls `optimizer.step(closure)` repeatedly inside `fit_from_config(...)`.
